In [1]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)

Scikit-learn version: 1.1.3


In [47]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoLarsCV

# --- 1. Make a tiny synthetic dataset ----------------------------
# 8 features, but the target y only depends on the first 3.
rng = np.random.RandomState(seed := 42)
n_samples, n_features = 100, 8
X = rng.randn(n_samples, n_features)
true_coef = np.array([3.0, -2.0, 1.5, 0, 0, 0, 0, 0])  # features 3-7 are noise
y = X @ true_coef + rng.randn(n_samples) * 0.5          # add a little noise

# --- 2. The two lines from your snippet --------------------------
folder = KFold(n_splits=5, shuffle=True, random_state=seed)  # 5-fold CV splitter
# NOTE: it's random_state (not random_states), and .fit(X, y) with a comma.
# normalize=True is deprecated in sklearn 1.1+ so we leave it out.
reg = LassoLarsCV(cv=folder).fit(X, y)

# --- 3. Inspect what Lasso learned -------------------------------
print("Chosen alpha (regularization strength):", round(reg.alpha_, 4))
print("Learned coefficients:")
for i, c in enumerate(reg.coef_):
    print(f"  feature {i}: {c:+.3f}   (true: {true_coef[i]:+.1f})")
print("R^2 on training data:", round(reg.score(X, y), 3))

Chosen alpha (regularization strength): 0.0017
Learned coefficients:
  feature 0: +2.999   (true: +3.0)
  feature 1: -1.906   (true: -2.0)
  feature 2: +1.484   (true: +1.5)
  feature 3: -0.017   (true: +0.0)
  feature 4: +0.000   (true: +0.0)
  feature 5: +0.000   (true: +0.0)
  feature 6: -0.045   (true: +0.0)
  feature 7: +0.013   (true: +0.0)
R^2 on training data: 0.982


/home/xps15/temp/sklearn/sk_venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:133: FutureWarning: The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsCV())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 
  warnings.warn(


In [48]:
reg.coef_

array([ 2.99948594, -1.90588702,  1.48357162, -0.01651417,  0.        ,
        0.        , -0.04461876,  0.01328947])

In [50]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoLarsCV

# --- 1. Load the Titanic dataset --------------------------------
titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame

# --- 2. Simple prep: predict who SURVIVED ------------------------
# Keep a few easy-to-understand columns and turn them into numbers.
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare"]].copy()
df["age"] = df["age"].fillna(df["age"].median())      # fill missing ages
df["fare"] = df["fare"].fillna(df["fare"].median())   # 1 passenger has no fare
df["sex"] = (df["sex"] == "female").astype(float)     # 1 = female, 0 = male

X = df[["pclass", "sex", "age", "sibsp", "parch", "fare"]].values.astype(float)
y = df["survived"].astype(float).values
feature_names = ["pclass", "sex=female", "age", "sibsp", "parch", "fare"]



/home/xps15/temp/sklearn/sk_venv/lib/python3.10/site-packages/sklearn/datasets/_arff_parser.py:195: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, ignore_index=True)


In [45]:
# --- 3. Same two lines as before ---------------------------------
folder = KFold(n_splits=5, shuffle=True, random_state=42)

In [46]:
regx = LassoLarsCV(cv=folder, normalize=True) # regx is an estimator for LassoLarsCV
reg=regx.fit(X, y)

In [35]:
(reg.alpha_, 
reg.coef_ , 
reg.score(X, y))

(4.969338598346617e-05,
 array([-1.59715543e-01,  4.94226680e-01, -5.37857375e-03, -4.30944935e-02,
        -2.36631391e-03,  3.53733001e-04]),
 0.3650543489068897)

In [36]:
# --- 4. Which features matter for survival? ----------------------
print("Chosen alpha:", round(reg.alpha_, 5))
print("\nCoefficients (bigger |value| = more influence on survival):")
for name, c in sorted(zip(feature_names, reg.coef_), key=lambda t: -abs(t[1])):
    print(f"  {name:12s} {c:+.5f}")
print("\nR^2:", round(reg.score(X, y), 3))

Chosen alpha: 5e-05

Coefficients (bigger |value| = more influence on survival):
  sex=female   +0.49423
  pclass       -0.15972
  sibsp        -0.04309
  age          -0.00538
  parch        -0.00237
  fare         +0.00035

R^2: 0.365


In [37]:
# --- 5. Peek inside the cross-validation -------------------------
# cv_alphas_  = the alpha candidates LARS produced
# mse_path_   = held-out error for each (fold, alpha); shape (n_folds, n_alphas)
print(f"{len(reg.cv_alphas_)} alpha candidates were tried across 5 folds\n")

mean_mse = reg.mse_path_.mean(axis=0)   # average error over the 5 folds

print("   alpha      mean held-out MSE")
for a, e in zip(reg.cv_alphas_[:10], mean_mse[:10]):   # show first 10
    marker = "  <-- chosen alpha_" if a == reg.alpha_ else ""
    print(f"  {a:.2e}      {e:.4f}{marker}")

print("\nalpha_ index in path:", list(reg.cv_alphas_).index(reg.alpha_),
      "of", len(reg.cv_alphas_))

31 alpha candidates were tried across 5 folds

   alpha      mean held-out MSE
  0.00e+00      0.1888
  1.43e-05      0.1696
  4.95e-05      0.1668
  4.97e-05      0.1534  <-- chosen alpha_
  1.94e-04      0.1961

alpha_ index in path: 3 of 31


In [39]:
# --- 6. Inference: predict survival for new passengers ------------
# The fitted model is just: score = intercept_ + sum(coef_ * feature)
# alpha is "used up" at fit time — it's baked into coef_, nothing extra needed.

# Two invented passengers, same columns as X:
# [pclass, sex=female, age, sibsp, parch, fare]
new_passengers = np.array([
    [1, 1, 25, 0, 0, 100],   # 1st-class woman, 25 y/o
    [3, 0, 40, 1, 2,  15],   # 3rd-class man, 40 y/o, with family
])

scores = reg.predict(new_passengers)   # estimated P(survival)
for desc, s in zip(["1st-class woman", "3rd-class man "], scores):
    print(f"{desc}: predicted survival probability = {s:.2f}")

# Sanity check: predict is literally the linear equation
manual = reg.intercept_ + new_passengers @ reg.coef_
print("\nSame result computed by hand:", manual.round(2))

1st-class woman: predicted survival probability = 0.98
3rd-class man : predicted survival probability = 0.01

Same result computed by hand: [0.98 0.01]


In [41]:
# --- 7. Watch Lasso select features as alpha grows ---------------
from sklearn.linear_model import Lasso

print(f"CV chose alpha = {reg.alpha_:.2e} — too weak to eliminate anything.\n")
print("alpha        " + "  ".join(f"{n:>10s}" for n in feature_names))
print("-" * 80)
for alpha in [reg.alpha_, 1e-3, 5e-3, 1e-2, 5e-2, 2e-1]:
    m = Lasso(alpha=alpha).fit(X, y)
    row = "  ".join(f"{c:>10.4f}" for c in m.coef_)
    n_kept = np.count_nonzero(m.coef_)
    print(f"{alpha:<12.2e} {row}   -> {n_kept}/6 features kept")

print("\nExactly-zero coefficient = feature ELIMINATED from the model.")

CV chose alpha = 4.97e-05 — too weak to eliminate anything.

alpha            pclass  sex=female         age       sibsp       parch        fare
--------------------------------------------------------------------------------
4.97e-05        -0.1615      0.4976     -0.0056     -0.0450     -0.0049      0.0004   -> 6/6 features kept
1.00e-03        -0.1597      0.4930     -0.0056     -0.0444     -0.0035      0.0004   -> 6/6 features kept
5.00e-03        -0.1522      0.4740     -0.0054     -0.0413     -0.0000      0.0005   -> 5/6 features kept
1.00e-02        -0.1422      0.4515     -0.0051     -0.0363     -0.0000      0.0006   -> 5/6 features kept
5.00e-02        -0.0617      0.2721     -0.0033     -0.0000     -0.0000      0.0014   -> 4/6 features kept
2.00e-01        -0.0000      0.0000     -0.0021     -0.0000      0.0000      0.0023   -> 2/6 features kept

Exactly-zero coefficient = feature ELIMINATED from the model.


In [43]:
# --- 8. Shrinkage shown side-by-side ------------------------------
from sklearn.linear_model import LinearRegression

ols = LinearRegression().fit(X, y)      # NO penalty at all (alpha = 0)

print(f"{'feature':12s} {'no penalty':>12s} {'Lasso':>12s}   what happened")
print("-" * 62)
for name, o, l in zip(feature_names, ols.coef_, reg.coef_):
    shrink = (1 - abs(l) / abs(o)) * 100
    print(f"{name:12s} {o:>12.5f} {l:>12.5f}   shrunk {shrink:5.1f}% toward zero")

print("\nREMOVED would look like this instead:")
print(f"parch with alpha=0.05 -> {Lasso(alpha=0.05).fit(X, y).coef_[4]}  (exactly 0.0)")

feature        no penalty        Lasso   what happened
--------------------------------------------------------------
pclass           -0.16155     -0.15972   shrunk   1.1% toward zero
sex=female        0.49789      0.49423   shrunk   0.7% toward zero
age              -0.00564     -0.00538   shrunk   4.6% toward zero
sibsp            -0.04502     -0.04309   shrunk   4.3% toward zero
parch            -0.00498     -0.00237   shrunk  52.5% toward zero
fare              0.00039      0.00035   shrunk  10.0% toward zero

REMOVED would look like this instead:
parch with alpha=0.05 -> -0.0  (exactly 0.0)


In [52]:
# --- 9. The sklearn >= 1.2 way: Pipeline + StandardScaler ---------
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# BEFORE (sklearn 1.1, normalize=True):
#   reg = LassoLarsCV(cv=folder, normalize=True).fit(X, y)
#
# AFTER: scaling becomes an explicit pipeline step.
pipe = make_pipeline(StandardScaler(), LassoLarsCV(cv=folder))
pipe.fit(X, y)

# predict / score work exactly as before — scaling is applied automatically:
print("predict:", pipe.predict(new_passengers).round(2))
print("R^2:    ", round(pipe.score(X, y), 3))

# The catch: fitted attributes now live on the inner step, not the pipe.
lasso_step = pipe.named_steps["lassolarscv"]   # or pipe[-1], or pipe.steps[-1][1]
print("alpha_:", round(lasso_step.alpha_, 5))

# coef_ is in SCALED units now — divide by the scale to compare with raw units:
print("coef_ (scaled units):", lasso_step.coef_.round(4))
print("coef_ (raw units):   ", (lasso_step.coef_ / pipe[0].scale_).round(4))
print("raw coef_ from cell 5:", reg.coef_.round(4), " <- should roughly match")

predict: [0.98 0.01]
R^2:     0.365
alpha_: 5e-05
coef_ (scaled units): [-0.1338  0.2366 -0.0694 -0.0449 -0.002   0.0183]
coef_ (raw units):    [-1.597e-01  4.942e-01 -5.400e-03 -4.310e-02 -2.400e-03  4.000e-04]
raw coef_ from cell 5: [ 2.9995 -1.9059  1.4836 -0.0165  0.      0.     -0.0446  0.0133]  <- should roughly match


/home/xps15/temp/sklearn/sk_venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:133: FutureWarning: The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsCV())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 
  warnings.warn(


In [53]:
# --- 10. Reproducing old normalize=True behavior -------------------
# Old normalize=True did: center columns, then scale each column to
# unit L2 norm, AND used a rescaled objective:
#     alpha_effective = alpha / sqrt(n_samples)
# sklearn's own warning suggests: StandardScaler(with_mean=False)
#     + alpha * sqrt(n_samples)
#
# Let's test how close each variant gets to the old model's coef_:

from sklearn.preprocessing import Normalizer  # noqa: F401 (just to show it exists)

n = X.shape[0]
old_coef = np.array([-0.15971554, 0.49422668, -0.00537857,
                     -0.04309449, -0.00236631, 0.00035373])  # from cell 5

# Variant A: plain StandardScaler (recommended, but NOT identical)
pipeA = make_pipeline(StandardScaler(), LassoLarsCV(cv=folder)).fit(X, y)
coefA = pipeA[-1].coef_ / pipeA[0].scale_

# Variant B: sklearn's suggested recipe (with_mean=False + alpha rescale)
pipeB = make_pipeline(StandardScaler(with_mean=False),
                      LassoLarsCV(cv=folder)).fit(X, y)
coefB = pipeB[-1].coef_ / pipeB[0].scale_

print(f"{'feature':12s} {'old norm=T':>12s} {'StdScaler':>12s} {'recipe B':>12s}")
print("-" * 52)
for name, o, a, b in zip(feature_names, old_coef, coefA, coefB):
    print(f"{name:12s} {o:>12.5f} {a:>12.5f} {b:>12.5f}")

print("\nMax |difference| vs old model:")
print(f"  plain StandardScaler : {np.abs(old_coef - coefA).max():.6f}")
print(f"  recipe B             : {np.abs(old_coef - coefB).max():.6f}")

feature        old norm=T    StdScaler     recipe B
----------------------------------------------------
pclass           -0.15972     -0.15972     -0.15972
sex=female        0.49423      0.49423      0.49423
age              -0.00538     -0.00538     -0.00538
sibsp            -0.04309     -0.04309     -0.04309
parch            -0.00237     -0.00237     -0.00237
fare              0.00035      0.00035      0.00035

Max |difference| vs old model:
  plain StandardScaler : 0.000000
  recipe B             : 0.000000


/home/xps15/temp/sklearn/sk_venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:133: FutureWarning: The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsCV())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 
  warnings.warn(
/home/xps15/temp/sklearn/sk_venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:133: FutureWarning: The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pip